In [1]:
!git clone https://github.com/cherchin/ranking-masters-project.git

Cloning into 'ranking-masters-project'...
remote: Enumerating objects: 1784, done.
remote: Counting objects: 100% (1784/1784), done.
remote: Compressing objects: 100% (1447/1447), done.
remote: Total 1784 (delta 286), reused 1746 (delta 264), pack-reused 0 (from 0)
Receiving objects: 100% (1784/1784), 10.90 MiB | 13.55 MiB/s, done.
Resolving deltas: 100% (286/286), done.


In [2]:
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
os.environ["GEMINI_MODEL"] = user_secrets.get_secret("GEMINI_MODEL")


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
!pip install -q -U transformers accelerate sentencepiece tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6

In [5]:
%cd /kaggle/working/ranking-masters-project

/kaggle/working/ranking-masters-project


In [6]:

import os
import subprocess

script = "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py"

processes = []

for batch, gpu in [(0, 0), (1, 1)]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)

    log_file = open(f"/kaggle/working/batch_{batch + 1}.log", "w")

    process = subprocess.Popen(
        ["python", script, "--batch", str(batch)],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    processes.append((process, log_file, batch))

for process, log_file, batch in processes:
    return_code = process.wait()
    log_file.close()
    print(f"Batch {batch + 1} finished with exit code {return_code}")

for batch in [1, 2]:
    print(f"\n===== BATCH {batch} =====")
    with open(f"/kaggle/working/batch_{batch}.log") as f:
        lines = f.readlines()
        print("".join(lines[-15:]))


Batch 1 finished with exit code 1
Batch 2 finished with exit code 1

===== BATCH 1 =====
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 449, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 401, in __get_result
    raise self._exception
  File "/usr/local/lib/python3.12/dist-packages/tenacity/__init__.py", line 473, in __call__
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1215, in _request_once
    errors.APIError.raise_for_response(response)
  File "/usr/local/lib/python3.12/dist-packages/google/genai/errors.py", line 134, in raise_for_response
    cls.raise_error(response.status_code, response_json, response)
  File "/usr/local/lib/python3.12/dist-packages/google/genai/errors.py", line 159, in raise_error
    raise ClientError(status_code, response_json, response)
google.genai.erro